In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv


In [2]:
import torch
device= "cuda" if torch.cuda.is_available() else "cpu"
device

'cuda'

# Text Cleaning


In [3]:
import re
train= pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv")
print(train.isnull().sum().sum())  #no nulls values

def clean_text(text):
    text = text.lower()
    text = re.sub(r"[^a-z0-9\-\./\s]", "", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

train["cleaned_prompt"] = train["prompt"].apply(clean_text)
for opt in ["A","B","C","D","E"]:
    train[f"cleaned_{opt}"] = train[f"{opt}"].apply(clean_text)
train.head(5)

0


,id,prompt,A,B,C,D,E,answer,cleaned_prompt,cleaned_A,cleaned_B,cleaned_C,cleaned_D,cleaned_E
0,1,Pick the best possible answer: What is Martin ...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B,pick the best possible answer what is martin h...,martin heidegger believes that humans exist wi...,martin heidegger believes that humans do not e...,martin heidegger does not believe in the exist...,martin heidegger believes that the relationshi...,martin heidegger believes that time is an illu...
1,2,What is accelerator-based light-ion fusion?,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,A,what is accelerator-based light-ion fusion,accelerator-based light-ion fusion is a techni...,accelerator-based light-ion fusion is a techni...,accelerator-based light-ion fusion is a techni...,accelerator-based light-ion fusion is a techni...,accelerator-based light-ion fusion is a techni...
2,3,Determine the correct option: What is the term...,Blueshifting,Redshifting,Reddening,Whitening,Yellowing,C,determine the correct option what is the term ...,blueshifting,redshifting,reddening,whitening,yellowing
3,4,Select the most accurate option: What is Marti...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B,select the most accurate option what is martin...,martin heidegger believes that humans exist wi...,martin heidegger believes that humans do not e...,martin heidegger does not believe in the exist...,martin heidegger believes that the relationshi...,martin heidegger believes that time is an illu...
4,5,Identify the correct statement: What is the co...,"Simultaneity is relative, meaning that two eve...","Simultaneity is relative, meaning that two eve...","Simultaneity is absolute, meaning that two eve...",Simultaneity is a concept that applies only to...,Simultaneity is a concept that applies only to...,A,identify the correct statement what is the con...,simultaneity is relative meaning that two even...,simultaneity is relative meaning that two even...,simultaneity is absolute meaning that two even...,simultaneity is a concept that applies only to...,simultaneity is a concept that applies only to...


In [4]:
train['answer'].value_counts()

answer
B    490
C    459
A    369
D    358
E    324
Name: count, dtype: int64

# Embeddings & Similarity 

In [13]:
import random
torch.manual_seed(42)
np.random.seed(42)
random.seed(42)

In [14]:
import nltk
from gensim.models import Word2Vec
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

nltk.download("punkt")
train["prompt_token"] = train["cleaned_prompt"].apply(nltk.word_tokenize)
for opt in ["A","B","C","D","E"]:
    train[f"{opt}_token"] = train[f"cleaned_{opt}"].apply(nltk.word_tokenize)

text = train["cleaned_prompt"].tolist()
for opt in ["A","B","C","D","E"]:
    text += train[f"cleaned_{opt}"].tolist()

tfidf = TfidfVectorizer()
tfidf.fit(text)

i = 0
p_vec = tfidf.transform([train.loc[i,"cleaned_prompt"]])
o_vec = [tfidf.transform([train.loc[i,f"cleaned_{opt}"]]) for opt in ["A","B","C","D","E"]]

cos_sim = [cosine_similarity(p_vec, opt)[0][0] for opt in o_vec]
print(cos_sim)

all_tokens = train["prompt_token"].tolist()
for opt in ["A","B","C","D","E"]:
    all_tokens += train[f"{opt}_token"].tolist()

w2v = Word2Vec(sentences=all_tokens, vector_size=100, window=5, min_count=1, workers=4)
def avg_vector(tokens, model):
    vecs = [model.wv[w] for w in tokens if w in model.wv]
    if len(vecs) > 0:
        return np.mean(vecs, axis=0) 
    else:
        return np.zeros(model.vector_size)
        
i = 0
p_avg = avg_vector(train.loc[i,"prompt_token"], w2v)
o_avg = [avg_vector(train.loc[i,f"{opt}_token"], w2v) for opt in ["A","B","C","D","E"]]

w2v_sim = [cosine_similarity([p_avg],[o])[0][0] for o in o_avg]
print(w2v_sim)

def predict_top3(prompt_tokens, option_tokens, model):
    p_vec = avg_vector(prompt_tokens, model)
    sim = [cosine_similarity([p_vec],[avg_vector(opt, model)])[0][0] for opt in option_tokens]
    ranked = np.argsort(sim)[::-1]
    return [chr(65 + idx) for idx in ranked[:3]]
    
def mapk(actual, predicted, k=3):
    score = 0.0
    n = len(actual)

    for i in range(n):
        a = actual[i]
        p = predicted[i]
        if a in p:
            idx = p.index(a)
            score += 1.0 / (idx + 1)
    return score / n
    
actual_ans = train["answer"].tolist()[:500]
pred_ans = []

for i in range(500):
    prompt_tokens = train.loc[i,"prompt_token"]
    option_tokens_list = [train.loc[i,f"{opt}_token"] for opt in ["A","B","C","D","E"]]
    preds = predict_top3(prompt_tokens, option_tokens_list, w2v)
    pred_ans.append(preds)

print(mapk(actual_ans, pred_ans))

[nltk_data] Downloading package punkt to /usr/share/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


[np.float64(0.16679528994578402), np.float64(0.18754309767044808), np.float64(0.4200856621712537), np.float64(0.37036798616566935), np.float64(0.10824678045170937)]
[np.float32(0.71562266), np.float32(0.72231543), np.float32(0.82484293), np.float32(0.7331053), np.float32(0.64712775)]
0.3386666666666669


# RoBERTa tokenization & training

In [15]:
from transformers import AutoTokenizer, RobertaTokenizer, RobertaForSequenceClassification, Trainer, TrainingArguments
from datasets import Dataset

tokenizer = AutoTokenizer.from_pretrained("roberta-base")
rows = []
for i in range(len(train)):
    prompt = train.loc[i,"cleaned_prompt"]
    for opt in ["A","B","C","D","E"]:
        text_pair = prompt + " " + train.loc[i,f"cleaned_{opt}"]
        label = 1 if train.loc[i,"answer"] == opt else 0
        rows.append({"text": text_pair, "label": label})

dataset = Dataset.from_list(rows)
def tokenize(batch):
    return tokenizer(batch["text"], padding="max_length", truncation=True)

dataset = dataset.map(tokenize, batched=True)
dataset = dataset.train_test_split(test_size=0.2, seed=42)
model = RobertaForSequenceClassification.from_pretrained("roberta-base", num_labels=2)
training_args = TrainingArguments(output_dir="./results", num_train_epochs=1, per_device_train_batch_size=8, per_device_eval_batch_size=8, logging_dir="./logs")
trainer = Trainer(model=model, args=training_args,train_dataset=dataset["train"],eval_dataset=dataset["test"])
trainer.train()

Map:   0%|          | 0/10000 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
classifier.out_proj.weight      | MISSING    | 
classifier.dense.weight         | MISSING    | 
classifier.out_proj.bias        | MISSING    | 
classifier.dense.bias           | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.
/usr/loca

Step,Training Loss
500,1.014679


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=500, training_loss=1.0146793823242188, metrics={'train_runtime': 448.9801, 'train_samples_per_second': 17.818, 'train_steps_per_second': 1.114, 'total_flos': 2104888442880000.0, 'train_loss': 1.0146793823242188, 'epoch': 1.0})

# Evaluation

In [16]:
pred = trainer.predict(dataset["test"])
probs = pred.predictions 
labels = pred.label_ids 
softmax = torch.nn.Softmax(dim=1)
probabilities = softmax(torch.tensor(probs)).numpy()

pred_opt = []
true_ans = []
rows_per_question = 5
for i in range(0, len(probs), rows_per_question):
    group = probs[i:i+rows_per_question, 1]
    ranked = np.argsort(-group)
    options = ["A","B","C","D","E"]
    ranked_options = [options[j] for j in ranked]
    pred_opt.append(ranked_options)
    correct_idx = np.argmax(labels[i:i+rows_per_question])
    true_ans.append(options[correct_idx])

def mapk(actual, predicted, k=3):
    score = 0.0
    for a, p in zip(actual, predicted):
        if a in p[:k]:
            score += 1.0 / (p.index(a)+1)
    return score/len(actual)
mapk(true_ans, pred_opt, k=3)

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


0.3775

# RAG Inference

In [17]:
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForSequenceClassification, pipeline

embedder = SentenceTransformer("all-MiniLM-L6-v2")
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
model = AutoModelForSequenceClassification.from_pretrained("bert-base-uncased", num_labels=2)
classifier = pipeline("zero-shot-classification", model=model, tokenizer=tokenizer)

def retrieve_context(question, opts, k=3):
    q_emb = embedder.encode([question], convert_to_numpy=True).astype("float32")
    opt_emb = embedder.encode(opts, convert_to_numpy=True).astype("float32")
    scores = np.dot(opt_emb, q_emb.T).flatten()
    ranked = np.argsort(-scores)
    return [opts[i] for i in ranked[:k]]
    
pred_opt = []
true_ans = []
opt = ["A","B","C","D","E"]
rows_per_question = 5
for i in range(0, len(dataset["test"]), rows_per_question):
    q_text = dataset["test"][i]["text"]
    opts = [dataset["test"][i+j]["text"] for j in range(rows_per_question)]
    retrieved = retrieve_context(q_text, opts, k=3)
    result = classifier(q_text, candidate_labels=retrieved)
    ranked_options = []
    for label in result["labels"]:
        if label in opts:
            ranked_options.append(options[opts.index(label)])
    pred_opt.append(ranked_options)
    correct_idx = np.argmax([dataset["test"][i+j]["label"] for j in range(rows_per_question)])
    true_ans.append(options[correct_idx])
mapk(true_ans, pred_opt, k=3)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
Failed to determine 'entailment' labe

0.3729166666666665

# FAISS Retrieval

In [18]:
!pip install faiss-cpu -q
import faiss

texts = [row["text"] for row in dataset["train"]]
embds = embedder.encode(texts)
dim = embds.shape[1]
index = faiss.IndexFlatL2(dim)
index.add(embds)
def retrieve_context(question,opts, k=3):
    q_emb = embedder.encode([question], convert_to_numpy=True).astype("float32")
    opt_emb = embedder.encode(opts, convert_to_numpy=True).astype("float32")
    dim = opt_emb.shape[1]
    index = faiss.IndexFlatL2(dim)
    index.add(opt_emb)
    D, I = index.search(q_emb, k)
    return [opts[idx] for idx in I[0]]

for i in range(0, len(dataset["test"]), rows_per_question):
    q_text = dataset["test"][i]["text"]
    opts = [dataset["test"][i+j]["text"] for j in range(rows_per_question)]
    retrieved = retrieve_context(q_text,opts, k=3)
    result = classifier(q_text, candidate_labels=retrieved)
    ranked_options = []
    for lbl in result["labels"]:
        if lbl in opts:
            ranked_options.append(options[opts.index(lbl)])
    pred_opt.append(ranked_options)
    correct_idx = np.argmax([dataset["test"][i+j]["label"] for j in range(rows_per_question)])
    true_ans.append(options[correct_idx])
mapk(true_ans, pred_opt, k=3)

0.37291666666666695

In [ ]:
sample= pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv')
sample.to_csv('submission.csv', index=False)